In [1]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv('perf_train.csv')
print(df.shape)
df.head()

(2000, 22)


,Employee_ID,Age,Gender,City,Education_Level,Department,Experience_Years,Monthly_Salary,Projects_Completed,Training_Hours,...,Job_Satisfaction,Manager_Rating,Overtime_Hours,Commute_Time_Min,Laptop_Issue_Count,Cafeteria_Rating,Internet_Stability,Last_Promotion_Years,Absenteeism_Days,Performance_Category
0,10001,49.0,Male,Bengaluru,Post Graduate,Sales,NaN,NaN,15.0,20.0,...,3.0,3.2,43.0,11.0,3.0,1.0,Good,2.1,1.0,Average Performer
1,10002,35.0,Female,Bengaluru,Post Graduate,Analytics,0.8,129555.0,15.0,163.0,...,4.0,2.3,43.0,128.0,6.0,5.0,Average,0.8,18.0,Low Performer
2,10003,28.0,Female,NaN,Graduate,Finance,12.7,55760.0,NaN,213.0,...,1.0,4.2,50.0,34.0,1.0,4.0,Poor,1.3,12.0,Average Performer
3,10004,41.0,Male,Hyderabad,Graduate,Sales,4.9,128174.0,NaN,209.0,...,2.0,4.5,78.0,168.0,7.0,5.0,Good,4.2,11.0,Average Performer
4,10005,39.0,Male,NaN,Graduate,Sales,1.2,145192.0,5.0,117.0,...,3.0,4.2,20.0,117.0,9.0,2.0,Excellent,2.5,24.0,Low Performer


In [3]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Employee_ID           2000 non-null   int64  
 1   Age                   1873 non-null   float64
 2   Gender                1934 non-null   str    
 3   City                  1789 non-null   str    
 4   Education_Level       1806 non-null   str    
 5   Department            1801 non-null   str    
 6   Experience_Years      1747 non-null   float64
 7   Monthly_Salary        1895 non-null   float64
 8   Projects_Completed    1712 non-null   float64
 9   Training_Hours        1702 non-null   float64
 10  Certifications        1695 non-null   float64
 11  Work_Life_Balance     1819 non-null   float64
 12  Job_Satisfaction      1804 non-null   float64
 13  Manager_Rating        1853 non-null   float64
 14  Overtime_Hours        1810 non-null   float64
 15  Commute_Time_Min      1805 non-n

Employee_ID               0
Age                     127
Gender                   66
City                    211
Education_Level         194
Department              199
Experience_Years        253
Monthly_Salary          105
Projects_Completed      288
Training_Hours          298
Certifications          305
Work_Life_Balance       181
Job_Satisfaction        196
Manager_Rating          147
Overtime_Hours          190
Commute_Time_Min        195
Laptop_Issue_Count       67
Cafeteria_Rating        283
Internet_Stability      343
Last_Promotion_Years    164
Absenteeism_Days        319
Performance_Category      0
dtype: int64

In [4]:
y = df['Performance_Category']
X = df.drop(['Performance_Category', 'Employee_ID'], axis=1)

cats = X.select_dtypes(include='object').columns.tolist()
X[cats] = X[cats].fillna('missing')
cat_idx = [X.columns.get_loc(c) for c in cats]

y.value_counts()

C:\Users\DELL\AppData\Local\Temp\ipykernel_6512\1620757990.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cats = X.select_dtypes(include='object').columns.tolist()


Performance_Category
Average Performer    1129
Low Performer         663
High Performer        208
Name: count, dtype: int64

In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [6]:
m = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05,
                       loss_function='MultiClass', eval_metric='Accuracy',
                       random_seed=42, verbose=100)
m.fit(X_tr, y_tr, cat_features=cat_idx, eval_set=(X_val, y_val), early_stopping_rounds=50)

0:	learn: 0.7687500	test: 0.7325000	best: 0.7325000 (0)	total: 205ms	remaining: 1m 42s
100:	learn: 0.8875000	test: 0.8850000	best: 0.8850000 (64)	total: 7.7s	remaining: 30.4s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.885
bestIteration = 64

Shrink model to first 65 iterations.


CatBoostClassifier(depth=6, eval_metric='Accuracy', iterations=500, learning_rate=0.05, loss_function='MultiClass', random_seed=42, verbose=100)

In [7]:
pred = m.predict(X_val).ravel()
print(accuracy_score(y_val, pred))
print(classification_report(y_val, pred))
confusion_matrix(y_val, pred)

0.885
                   precision    recall  f1-score   support

Average Performer       0.85      0.99      0.91       226
   High Performer       1.00      0.49      0.66        41
    Low Performer       0.95      0.83      0.89       133

         accuracy                           0.89       400
        macro avg       0.93      0.77      0.82       400
     weighted avg       0.90      0.89      0.88       400



array([[223,   0,   3],
       [ 18,  20,   3],
       [ 22,   0, 111]])

In [8]:
# try a deeper one
m2 = CatBoostClassifier(iterations=500, depth=8, learning_rate=0.05,
                        loss_function='MultiClass', eval_metric='Accuracy',
                        random_seed=42, verbose=0)
m2.fit(X_tr, y_tr, cat_features=cat_idx, eval_set=(X_val, y_val), early_stopping_rounds=50)
print(accuracy_score(y_val, m2.predict(X_val).ravel()))

0.885


In [9]:
pd.Series(m.feature_importances_, index=X.columns).sort_values(ascending=False)

Projects_Completed      45.991022
Experience_Years        36.390057
Manager_Rating          12.637443
Education_Level          0.533894
Monthly_Salary           0.519723
Certifications           0.519203
Department               0.433522
Work_Life_Balance        0.378571
Age                      0.366211
Gender                   0.357900
Commute_Time_Min         0.325043
City                     0.281325
Training_Hours           0.216207
Laptop_Issue_Count       0.213242
Cafeteria_Rating         0.210111
Job_Satisfaction         0.154780
Absenteeism_Days         0.145706
Internet_Stability       0.131460
Overtime_Hours           0.127167
Last_Promotion_Years     0.067412
dtype: float64

In [10]:
final = CatBoostClassifier(iterations=m.get_best_iteration() + 1, depth=6, learning_rate=0.05,
                           loss_function='MultiClass', random_seed=42, verbose=0)
final.fit(X, y, cat_features=cat_idx)
final.save_model('employee_performance_catboost.cbm')

In [11]:
test = pd.read_csv('perf_test.csv')
Xt = test[X.columns].copy()
Xt[cats] = Xt[cats].fillna('missing')

out = pd.DataFrame({'Employee_ID': test['Employee_ID'],
                    'Performance_Category': final.predict(Xt).ravel()})
out.to_csv('Model_Result.csv', index=False)
out.head()

,Employee_ID,Performance_Category
0,12001,Average Performer
1,12002,Average Performer
2,12003,Low Performer
3,12004,Low Performer
4,12005,Average Performer
